In [ ]:
import re
from pathlib import Path

import numpy as np
import torch
import soundfile as sf
import openl3
from tqdm import tqdm
import pkg_resources

AUDIO_PATH = Path("C:/Users/cathy/Downloads/48k(+5)/48k(+5)/audio_1000_balanced_pitch_5_48k")
OUTPUT_DIR = Path("C:/Users/cathy/Downloads/OpenL3_embeddings/5")
OUTPUT_DIR.mkdir(exist_ok=True)

def getID(filename) -> str:
    stem = Path(filename).stem
    m = re.search(r"DB_(?:training|test|validation)-(\d+)_(.+)$", stem)
    if not m:
        raise ValueError(f"Could not extract ID from filename: {filename}")
    return m.group(2), m.group(1)

In [ ]:
all_embeddings = {}

for file in tqdm(AUDIO_PATH.iterdir(), desc="embeddings"):
    if file.name.startswith("."):
        continue

    file_id, instrument_id = getID(file.stem)

    out_path = OUTPUT_DIR / f"{file_id}.pt"

    if out_path.exists():
        continue

    audio, sr = sf.read(file)

    emb, ts = openl3.get_audio_embedding(
        audio,
        sr,
        content_type="music",
        embedding_size=512
    )

    emb_mean = np.mean(emb, axis=0)
    emb_tensor = torch.tensor(emb_mean)

    record = {
        "file_id": file_id,
        "source_file": file.name,
        "sample_rate": sr,
        "embedding": emb_tensor,
        "instrument_id": instrument_id,
    }

    # save per file
    torch.save(record, out_path)

    # collect into dict
    all_embeddings[file_id] = record

# save combined file
torch.save(all_embeddings, OUTPUT_DIR / "all_openl3_embeddings.pt")

print("✅ Saved combined file:", OUTPUT_DIR / "all_openl3_embeddings.pt")